In [ ]:
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Create connection to database
conn = sqlite3.connect('data.sqlite')

In [ ]:
query_california = """
SELECT *
FROM customers
WHERE state = 'CA' AND creditLimit > 25000;
"""
ca_customers_df = pd.read_sql(query_california, conn)
ca_customers_df

In [ ]:
query_international = """
SELECT *
FROM customers
WHERE country != 'USA' AND customerName LIKE '%Collect%';
"""
intl_collect_df = pd.read_sql(query_international, conn)
intl_collect_df

In [ ]:
query_usa_credit = """
SELECT state, AVG(creditLimit) AS avg_credit_limit
FROM customers
WHERE country = 'USA'
GROUP BY state
ORDER BY avg_credit_limit DESC;
"""
usa_credit_df = pd.read_sql(query_usa_credit, conn)
usa_credit_df

In [ ]:
plt.figure(figsize=(12, 6))
sns.barplot(data=usa_credit_df, x='state', y='avg_credit_limit', palette='viridis')
plt.title('Average Customer Credit Limit by US State')
plt.xlabel('State')
plt.ylabel('Average Credit Limit ($)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
query_top_customers = """
SELECT c.customerName, SUM(p.amount) AS total_payments
FROM customers c
JOIN payments p ON c.customerNumber = p.customerNumber
GROUP BY c.customerName
ORDER BY total_payments DESC
LIMIT 10;
"""
top_customers_df = pd.read_sql(query_top_customers, conn)
top_customers_df

In [ ]:
plt.figure(figsize=(10, 6))
sns.barplot(data=top_customers_df, x='total_payments', y='customerName', palette='Blues_r')
plt.title('Top 10 Customers by Total Payments Made')
plt.xlabel('Total Payments ($)')
plt.ylabel('Customer Name')
plt.tight_layout()
plt.show()

In [ ]:
query_cust_products = """
SELECT c.customerName, pr.productName, 
       SUM(od.quantityOrdered) AS total_quantity, 
       SUM(od.quantityOrdered * od.priceEach) AS total_amount
FROM customers c
JOIN orders o ON c.customerNumber = o.customerNumber
JOIN orderdetails od ON o.orderNumber = od.orderNumber
JOIN products pr ON od.productCode = pr.productCode
GROUP BY c.customerName, pr.productName
HAVING SUM(od.quantityOrdered) >= 10
ORDER BY total_amount ASC;
"""
cust_products_df = pd.read_sql(query_cust_products, conn)
cust_products_df

In [ ]:
query_product_analysis = """
SELECT pl.productLine, 
       COUNT(DISTINCT pr.productCode) AS total_products, 
       SUM(od.quantityOrdered) AS total_quantity_ordered
FROM productlines pl
JOIN products pr ON pl.productLine = pr.productLine
JOIN orderdetails od ON pr.productCode = od.productCode
GROUP BY pl.productLine;
"""
product_analysis_df = pd.read_sql(query_product_analysis, conn)
product_analysis_df

In [ ]:
plt.figure(figsize=(10, 6))
sns.scatterplot(data=product_analysis_df, x='total_products', y='total_quantity_ordered', s=200, color='crimson')

# Annotate points with product line names for clarity
for i in range(product_analysis_df.shape[0]):
    plt.text(product_analysis_df.total_products[i] + 0.5, 
             product_analysis_df.total_quantity_ordered[i], 
             product_analysis_df.productLine[i], 
             fontsize=10)

plt.title('Product Line Analysis: Variety vs. Order Volume')
plt.xlabel('Number of Unique Products in Line')
plt.ylabel('Total Quantity Ordered')
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()

In [ ]:
query_remote = """
SELECT e.firstName, e.lastName, e.jobTitle, o.city AS office_city,
       sup.firstName AS supervisor_firstName, sup.lastName AS supervisor_lastName
FROM employees e
JOIN offices o ON e.officeCode = o.officeCode
LEFT JOIN employees sup ON e.reportsTo = sup.employeeNumber
WHERE e.officeCode IN (
    SELECT officeCode
    FROM employees
    GROUP BY officeCode
    HAVING COUNT(employeeNumber) < 5
);
"""
remote_offices_df = pd.read_sql(query_remote, conn)
remote_offices_df

In [ ]:
conn.close()

In [ ]:
# Connection to movies database
conn4 = sqlite3.connect('im.db')

# Explore query: Finding the highest-rated genres with substantial volume (2010-2019)
explore_query = """
SELECT mb.genres, 
       COUNT(mb.movie_id) as total_movies, 
       AVG(mr.averagerating) as avg_rating
FROM movie_basics mb
JOIN movie_ratings mr ON mb.movie_id = mr.movie_id
WHERE mb.start_year BETWEEN 2010 AND 2019
  AND mb.genres IS NOT NULL
GROUP BY mb.genres
HAVING total_movies > 50
ORDER BY avg_rating DESC
LIMIT 10;
"""
explore_df = pd.read_sql(explore_query, conn4)

# Visualization for Deliverable 2
plt.figure(figsize=(10,6))
sns.barplot(data=explore_df, x='avg_rating', y='genres', palette='mako')
plt.title('Top Rated Movie Genres (2010-2019, Min. 50 Movies)')
plt.xlabel('Average Rating')
plt.ylabel('Genre Category')
plt.tight_layout()
plt.show()

conn4.close()